In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls "/content/drive/MyDrive"

 043540042339_copy.pdf
'07.04.2025| Embedded Training| 2 Weeks|Attendance Sheet.gsheet'
'1000 Startups - SNS - GenAI.png'
 1735795950344.jfif
 1.approj
'2024 Internal Marks'
 2025-07-02T0445_Grades-ACCGAIv1EN-LTI13-126118.csv
 2025-07-02T0445_Grades-ACCGAIv1EN-LTI13-126118.gsheet
 20250807_122907.mp4
'2026 Engg_08 Jan Assessement (1).xlsx'
'2026 Engg_08 Jan Assessement.xlsx'
"22_07_2025 Today's attendees.xlsx"
'23ECB222-DPCO-IAE-Result Analysis.gsheet'
'23ECT201-EEI-IAE-1 QP SET B.docx'
'23ECT201-EEI-IAE-3 QP SET A.docx'
'Adaptive Steering Control System for Autonomous Vehicles on Uneven Terrain.gdoc'
'Additional Content for Services & Sensors Actuators.gdoc'
'AI (2).gdoc'
'AI (2).pdf'
 AI-Accelerators-and-Neural-Processing-Units.pptx
 AI.gdoc
'AI & ML BlackBelt Plus - Cohort LS2 Analytics Vidya List.gsheet'
 AI.pdf
'Air Quality Index Project Guide.gdoc'
'Altium Designer Course Modules & Faculty List.gdoc'
'April 2025 OKR'
'ARP & BOG (1) (1).xlsx'
'ARP & BOG (1).xlsx'
'ATAL FDP Certifi

In [3]:
DATA_PATH = "/content/drive/MyDrive/BCI_IV_2a"

In [4]:
!ls /content/drive/MyDrive/BCI_IV_2a

A01E.gdf  A02T.gdf  A04E.gdf  A05T.gdf	A07E.gdf  A08T.gdf
A01T.gdf  A03E.gdf  A04T.gdf  A06E.gdf	A07T.gdf  A09E.gdf
A02E.gdf  A03T.gdf  A05E.gdf  A06T.gdf	A08E.gdf  A09T.gdf


In [5]:
!pip install mne scipy torch torchvision scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 79.6 MB/s eta 0:00:00


In [6]:
import os
import numpy as np
import mne
from scipy.io import loadmat

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, cohen_kappa_score

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [7]:
train_file = "/content/drive/MyDrive/BCI_IV_2a/A01T.gdf"
test_file  = "/content/drive/MyDrive/BCI_IV_2a/A01E.gdf"

In [8]:
def load_bci_data(file_path):

    raw = mne.io.read_raw_gdf(file_path, preload=True, verbose=False)

    # Remove EOG channels
    raw.pick_types(eeg=True)

    # Bandpass filter (standard for MI)
    raw.filter(8., 30., fir_design='firwin')

    events, event_id = mne.events_from_annotations(raw)

    event_map = {
        '769':0,  # left hand
        '770':1,  # right hand
        '771':2,  # feet
        '772':3   # tongue
    }

    selected_events = []
    labels = []

    for ev in events:
        code = str(ev[2])
        if code in event_map:
            selected_events.append(ev)
            labels.append(event_map[code])

    selected_events = np.array(selected_events)
    labels = np.array(labels)

    epochs = mne.Epochs(
        raw,
        selected_events,
        tmin=3,
        tmax=6,
        baseline=None,
        preload=True
    )

    data = epochs.get_data()   # shape (trials, channels, samples)

    return data, labels

In [21]:
def load_bci_data(file_path):

    raw = mne.io.read_raw_gdf(file_path, preload=True, verbose=False)

    # Keep only EEG channels (removes EOG automatically)
    # Remove EOG channels explicitly
    eog_channels = ['EOG-left', 'EOG-central', 'EOG-right']

    existing_eog = [ch for ch in eog_channels if ch in raw.ch_names]

    raw.drop_channels(existing_eog)

    # Bandpass filter for motor imagery
    raw.filter(8., 30., fir_design='firwin', verbose=False)

    # Extract events
    events, event_dict = mne.events_from_annotations(raw)

    # Motor imagery event mapping
    event_map = {
        '769':0,  # left hand
        '770':1,  # right hand
        '771':2,  # feet
        '772':3   # tongue
    }

    selected_events = []
    labels = []

    for ev in events:
        event_code = ev[2]

        # convert event code → annotation name
        event_name = None
        for k, v in event_dict.items():
            if v == event_code:
                event_name = k
                break

        if event_name in event_map:
            selected_events.append(ev)
            labels.append(event_map[event_name])

    selected_events = np.array(selected_events, dtype=int)
    labels = np.array(labels, dtype=int)

    print("Motor imagery trials found:", len(labels))

    # Epoch extraction (3–6s motor imagery window)
    epochs = mne.Epochs(
        raw,
        selected_events,
        tmin=3,
        tmax=6,
        baseline=None,
        preload=True,
        verbose=False
    )

    data = epochs.get_data()

    # Force correct length
    data = data[:, :, :750]

    # Match labels to actual extracted trials
    labels = labels[:data.shape[0]]

    return data, labels

In [22]:
X, y = load_bci_data(train_file)

print("Dataset shape:", X.shape)
print("Labels:", np.unique(y))

/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('771'), np.str_('772')]
Motor imagery trials found: 288
Dataset shape: (287, 22, 750)
Labels: [0 1 2 3]


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (229, 22, 750)
Test shape: (58, 22, 750)


In [24]:
mean = np.mean(X_train)
std = np.std(X_train)

X_train = (X_train - mean) / std
X_test  = (X_test - mean) / std

In [25]:
print(np.mean(X_train), np.std(X_train))

4.2057134538241735e-18 1.0000000000000004


In [26]:
X_train = X_train[:, np.newaxis, :, :]
X_test  = X_test[:, np.newaxis, :, :]

print(X_train.shape)

(229, 1, 22, 750)


In [27]:
import torch
from torch.utils.data import Dataset

class EEGDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [41]:
from torch.utils.data import DataLoader

batch_size = 16

train_loader = DataLoader(
    EEGDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    EEGDataset(X_test, y_test),
    batch_size=batch_size,
    shuffle=False
)

In [42]:
print(len(train_loader))

15


In [43]:
import torch.nn as nn

class EEGNet(nn.Module):

    def __init__(self, num_classes=4):
        super().__init__()

        F1 = 8
        D = 2
        F2 = F1 * D

        self.firstconv = nn.Sequential(
            nn.Conv2d(1, F1, (1,64), padding=(0,32), bias=False),
            nn.BatchNorm2d(F1)
        )

        self.depthwise = nn.Sequential(
            nn.Conv2d(F1, F1*D, (22,1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D),
            nn.ELU(),
            nn.AvgPool2d((1,4)),
            nn.Dropout(0.5)
        )

        self.separable = nn.Sequential(
            nn.Conv2d(F1*D, F1*D, (1,16), padding=(0,8), groups=F1*D, bias=False),
            nn.Conv2d(F1*D, F2, 1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1,8)),
            nn.Dropout(0.5)
        )

        self.fc = nn.Linear(F2 * (750 // 32), num_classes)

    def forward(self, x):

        x = self.firstconv(x)
        x = self.depthwise(x)
        x = self.separable(x)

        x = x.view(x.size(0), -1)

        return self.fc(x)

In [44]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EEGNet().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=50,
    gamma=0.5
)

In [45]:
print(model)

EEGNet(
  (firstconv): Sequential(
    (0): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (depthwise): Sequential(
    (0): Conv2d(8, 16, kernel_size=(22, 1), stride=(1, 1), groups=8, bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
    (3): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
    (4): Dropout(p=0.5, inplace=False)
  )
  (separable): Sequential(
    (0): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=16, bias=False)
    (1): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ELU(alpha=1.0)
    (4): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
    (5): Dropout(p=0.5, inplace=False)
  )
  (fc): Linear(in_features=368, out_fea

In [46]:
epochs = 200

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    print(f"Epoch {epoch+1}/{epochs} Loss:{total_loss:.4f}")

Epoch 1/200 Loss:21.1814
Epoch 2/200 Loss:20.5104
Epoch 3/200 Loss:20.8341
Epoch 4/200 Loss:20.3619
Epoch 5/200 Loss:20.0883
Epoch 6/200 Loss:19.8442
Epoch 7/200 Loss:19.4956
Epoch 8/200 Loss:19.4643
Epoch 9/200 Loss:19.4432
Epoch 10/200 Loss:18.5155
Epoch 11/200 Loss:18.0405
Epoch 12/200 Loss:17.0202
Epoch 13/200 Loss:16.9835
Epoch 14/200 Loss:16.6946
Epoch 15/200 Loss:16.4782
Epoch 16/200 Loss:16.5295
Epoch 17/200 Loss:15.4884
Epoch 18/200 Loss:15.1386
Epoch 19/200 Loss:15.2545
Epoch 20/200 Loss:14.9287
Epoch 21/200 Loss:14.5442
Epoch 22/200 Loss:14.1550
Epoch 23/200 Loss:13.9833
Epoch 24/200 Loss:14.4329
Epoch 25/200 Loss:13.7755
Epoch 26/200 Loss:13.9093
Epoch 27/200 Loss:13.4188
Epoch 28/200 Loss:12.6156
Epoch 29/200 Loss:13.8578
Epoch 30/200 Loss:13.5596
Epoch 31/200 Loss:13.4390
Epoch 32/200 Loss:12.1630
Epoch 33/200 Loss:13.1059
Epoch 34/200 Loss:13.1560
Epoch 35/200 Loss:13.0735
Epoch 36/200 Loss:12.1515
Epoch 37/200 Loss:12.0055
Epoch 38/200 Loss:12.3557
Epoch 39/200 Loss:12.

In [48]:
from sklearn.metrics import accuracy_score, cohen_kappa_score

model.eval()

preds = []
truth = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        pred = torch.argmax(outputs, axis=1).cpu().numpy()

        preds.extend(pred)
        truth.extend(y_batch.numpy())

acc = accuracy_score(truth, preds)
kappa = cohen_kappa_score(truth, preds)

print("Test Accuracy:", acc)
print("Kappa:", kappa)

Test Accuracy: 0.4827586206896552
Kappa: 0.31576877703499817
